# Chapter 8 — Image Classification (ConvNets): **from scratch AND with Keras**

Maps to Chollet Ch.8. ConvNets are *the* model for images. Your lab flags **CNN-from-scratch** as important
but never coded it — so Part A builds the convolution + pooling operations in pure NumPy (and checks them
against Keras), and Part B uses the real `Conv2D`/`MaxPooling2D` layers + small-data tricks (augmentation,
transfer learning).

### Why ConvNets beat Dense layers on images
- **Local patterns**: a conv filter looks at a small window (e.g. 3×3), learning edges/textures — not all
  pixels at once like `Dense`.
- **Translation invariance**: a pattern learned in one corner is recognized anywhere → far fewer samples needed.
- **Spatial hierarchy**: layer 1 learns edges → layer 2 combines them into motifs → layer 3 into objects.

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np, matplotlib.pyplot as plt
from keras import layers


---
# Part A — The convolution operation, from scratch

## A1. 2D convolution by hand
A conv layer **slides a kernel** over the image, computing a dot product at each position. Inputs/outputs are
rank-3 **feature maps** `(H, W, channels)`. A kernel has shape `(kh, kw, C_in, C_out)`: it mixes all input
channels and produces `C_out` output channels (filters). Output (valid, stride 1) shrinks by `k-1`.

In [ ]:
def conv2d_forward(x, kernel, bias):
    # x:(H,W,Cin)  kernel:(kh,kw,Cin,Cout)  bias:(Cout,)  -> (H-kh+1, W-kw+1, Cout)
    H, W, Cin = x.shape
    kh, kw, _, Cout = kernel.shape
    oh, ow = H - kh + 1, W - kw + 1
    out = np.zeros((oh, ow, Cout), dtype="float32")
    for i in range(oh):
        for j in range(ow):
            patch = x[i:i+kh, j:j+kw, :]              # the receptive field (kh,kw,Cin)
            for co in range(Cout):
                out[i, j, co] = np.sum(patch * kernel[:, :, :, co]) + bias[co]
    return out

# numerically verify against Keras Conv2D (same weights -> same output)
x = np.random.RandomState(0).randn(8, 8, 3).astype("float32")
conv = layers.Conv2D(4, 3, padding="valid"); _ = conv(x[None])   # build to create weights
K, b = conv.get_weights()
diff = np.abs(conv2d_forward(x, K, b) - conv(x[None]).numpy()[0]).max()
print(f"max |mine - Keras| = {diff:.2e}  -> our convolution matches Keras")


## A2. A convolution *is* a pattern detector — hand-set edge kernels
Set the kernel by hand (no learning) to a vertical / horizontal edge detector and look at the response map.
A trained ConvNet learns thousands of such kernels automatically.

In [ ]:
(imgs, _), _ = keras.datasets.mnist.load_data()
digit = (imgs[7].astype("float32")/255)[:, :, None]      # one digit, shape (28,28,1)

vert = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype="float32")   # detects vertical edges
horiz = vert.T                                                   # detects horizontal edges
kernel = np.stack([vert, horiz], axis=-1)[:, :, None, :]         # (3,3,1,2)
resp = conv2d_forward(digit, kernel, np.zeros(2))

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(digit[:, :, 0], cmap="gray");   ax[0].set_title("input digit")
ax[1].imshow(resp[:, :, 0], cmap="gray");    ax[1].set_title("vertical-edge response")
ax[2].imshow(resp[:, :, 1], cmap="gray");    ax[2].set_title("horizontal-edge response")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## A3. Max pooling by hand
Pooling **downsamples** feature maps: take the max in each (usually 2×2) window. It throws away precise
position (adding robustness) and forces deeper layers to see a larger fraction of the image. No learnable
weights.

In [ ]:
def maxpool2d_forward(x, size=2, stride=2):
    H, W, C = x.shape
    oh, ow = (H-size)//stride + 1, (W-size)//stride + 1
    out = np.zeros((oh, ow, C), dtype="float32")
    for i in range(oh):
        for j in range(ow):
            out[i, j, :] = x[i*stride:i*stride+size, j*stride:j*stride+size, :].max(axis=(0,1))
    return out

pool = layers.MaxPooling2D(2)
diff = np.abs(maxpool2d_forward(x, 2, 2) - pool(x[None]).numpy()[0]).max()
print(f"max |mine - Keras pool| = {diff:.2e}  (exact match)")


---
# Part B — ConvNets with Keras

## B1. A real ConvNet on MNIST → ~99% (vs 97.8% for the Dense net of Ch.2)
A stack of `Conv2D` + `MaxPooling2D`, then `GlobalAveragePooling2D` to flatten, then a `Dense` classifier.
Watch the pattern: **depth (channels) goes up, spatial size goes down.** Input shape is
`(H, W, channels)`.

In [ ]:
(Xtr, ytr), (Xte, yte) = keras.datasets.mnist.load_data()
Xtr = Xtr.reshape(-1,28,28,1).astype("float32")/255
Xte = Xte.reshape(-1,28,28,1).astype("float32")/255

inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu")(inputs)   # (26,26,32)
x = layers.MaxPooling2D(2)(x)                          # (13,13,32)
x = layers.Conv2D(64, 3, activation="relu")(x)         # (11,11,64)
x = layers.MaxPooling2D(2)(x)                          # (5,5,64)
x = layers.Conv2D(64, 3, activation="relu")(x)         # (3,3,64)
x = layers.GlobalAveragePooling2D()(x)                 # (64,)
outputs = layers.Dense(10, activation="softmax")(x)
cnn = keras.Model(inputs, outputs)
cnn.summary()


In [ ]:
cnn.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(Xtr, ytr, epochs=5, batch_size=64, validation_split=0.1, verbose=0)
print("CNN test acc:", round(cnn.evaluate(Xte, yte, verbose=0, return_dict=True)["accuracy"], 4))


## B2. Output size, padding & strides (know the formula)
For input size `W`, kernel `K`, padding `P`, stride `S`:
$$\text{out} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$
- `padding="valid"` (default): `P=0`, output shrinks by `K-1`.
- `padding="same"`: pads so output size = input size (stride 1).
- `stride>1`: downsamples (alternative to pooling).

In [ ]:
def out_size(W, K, P, S): return (W - K + 2*P)//S + 1
print("28, k3, valid (P=0,S=1):", out_size(28,3,0,1))   # 26
print("28, k3, same  (P=1,S=1):", out_size(28,3,1,1))   # 28
print("28, k3, stride2(P=0,S=2):", out_size(28,3,0,2))  # 13
for pad in ["valid", "same"]:
    o = layers.Conv2D(8, 3, padding=pad, strides=1)(np.zeros((1,28,28,1),"float32"))
    print(f"Keras padding={pad}: {o.shape}")


## B3. Small data → data augmentation
With few images, ConvNets overfit fast. **Augmentation** generates plausible variants (flips, rotations,
zooms) on the fly, so the model never sees the exact same image twice. Add it as the first layers (active
only during training).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

(cifar_x, _), _ = keras.datasets.cifar10.load_data()
sample = cifar_x[:1].astype("float32")
fig, ax = plt.subplots(1, 5, figsize=(12, 3))
ax[0].imshow(sample[0].astype("uint8")); ax[0].set_title("original")
for i in range(1, 5):
    ax[i].imshow(np.clip(data_augmentation(sample)[0].numpy(), 0, 255).astype("uint8"))
    ax[i].set_title("augmented")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## B4. Transfer learning — the single biggest win on small image data
Don't train from scratch — reuse a model **pretrained on ImageNet** (millions of images). Two modes:
1. **Feature extraction**: freeze the pretrained convolutional **base**, train only a new classifier head.
2. **Fine-tuning**: after that, unfreeze the top base layers and train them too at a *low* learning rate.

Demo: classify two CIFAR-10 classes using a frozen **MobileNetV2** base (downloads ImageNet weights).

In [ ]:
(cx, cy), (cx_te, cy_te) = keras.datasets.cifar10.load_data()
# binary subset: class 0 (airplane) vs 1 (automobile), small + resized to 96x96 for MobileNetV2
def subset(x, y, n=1000):
    m = (y[:,0] < 2); x, y = x[m][:n], y[m][:n,0].astype("float32")
    x = keras.ops.image.resize(x.astype("float32"), (96,96))
    return np.array(x), y
Xtr2, ytr2 = subset(cx, cy, 1500); Xte2, yte2 = subset(cx_te, cy_te, 500)

base = keras.applications.MobileNetV2(input_shape=(96,96,3), include_top=False, weights="imagenet")
base.trainable = False                                    # FREEZE the pretrained base

inputs = keras.Input((96,96,3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)   # match pretraining preprocessing
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
tl_model = keras.Model(inputs, outputs)
tl_model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
tl_model.fit(Xtr2, ytr2, epochs=3, batch_size=32, validation_split=0.2, verbose=0)
print("feature-extraction test acc:",
      round(tl_model.evaluate(Xte2, yte2, verbose=0, return_dict=True)["accuracy"], 3))


In [ ]:
# Fine-tuning: unfreeze the top of the base, continue at a LOW learning rate
base.trainable = True
for layer in base.layers[:-20]:        # keep early (generic) layers frozen
    layer.trainable = False
tl_model.compile(keras.optimizers.Adam(1e-5), "binary_crossentropy", metrics=["accuracy"])
tl_model.fit(Xtr2, ytr2, epochs=2, batch_size=32, validation_split=0.2, verbose=0)
print("after fine-tuning test acc:",
      round(tl_model.evaluate(Xte2, yte2, verbose=0, return_dict=True)["accuracy"], 3))


---
# ✍️ PROBLEMS

### P1 — Convolution from scratch, extended
Add `stride` and `padding="same"` support to `conv2d_forward`. Verify against
`layers.Conv2D(..., strides=s, padding="same")` for a few (k, s) combos. Then implement
`avgpool2d_forward` and compare to `layers.AveragePooling2D`.

In [ ]:
# TODO


### P2 — Build & beat on Fashion-MNIST
Load `keras.datasets.fashion_mnist`. Build a ConvNet (Conv→Pool ×3 → GAP → Dense). Beat 90% test accuracy.
Then add `data_augmentation` (RandomFlip/Rotation) — does it help or hurt on this dataset, and why?

In [ ]:
# TODO


### P3 — The receptive field
For the B1 model, compute by hand what region of the *input* image a single neuron in the **last** Conv
layer depends on (its receptive field). Verify your reasoning with the output-size formula at each layer.

In [ ]:
# TODO


### P4 — Transfer learning on 3 classes
Extend B4 to 3 CIFAR-10 classes (softmax + sparse_categorical_crossentropy). Compare: (a) a small ConvNet
trained from scratch, (b) frozen-base feature extraction, (c) feature extraction + fine-tuning. Rank them.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — ConvNet classifier (the standard image template)

In [ ]:
import keras
from keras import layers
inputs = keras.Input(shape=(H, W, C))
x = layers.Rescaling(1./255)(inputs)              # if inputs are 0..255
x = layers.Conv2D(32, 3, activation="relu")(x); x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64, 3, activation="relu")(x); x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128,3, activation="relu")(x); x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
# binary: Dense(1,"sigmoid")+binary_crossentropy ; multiclass: Dense(C,"softmax")+sparse_categorical_crossentropy
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])


### T2 — Data augmentation block

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
# put right after Input:  x = data_augmentation(inputs)


### T3 — Transfer learning (feature extraction → fine-tuning)

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(96,96,3), include_top=False, weights="imagenet")
base.trainable = False
inputs = keras.Input((96,96,3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=10)         # 1) feature extraction
# 2) fine-tune:
base.trainable = True
for l in base.layers[:-20]: l.trainable = False
model.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=5)


### T4 — Conv & pool from scratch (NumPy)

In [ ]:
import numpy as np
def conv2d_forward(x, kernel, bias):   # x:(H,W,Cin) kernel:(kh,kw,Cin,Cout) bias:(Cout,)
    H,W,Cin=x.shape; kh,kw,_,Cout=kernel.shape; oh,ow=H-kh+1,W-kw+1
    out=np.zeros((oh,ow,Cout),"float32")
    for i in range(oh):
        for j in range(ow):
            patch=x[i:i+kh,j:j+kw,:]
            for co in range(Cout): out[i,j,co]=np.sum(patch*kernel[:,:,:,co])+bias[co]
    return out
def maxpool2d_forward(x,size=2,stride=2):
    H,W,C=x.shape; oh=(H-size)//stride+1; ow=(W-size)//stride+1
    out=np.zeros((oh,ow,C),"float32")
    for i in range(oh):
        for j in range(ow):
            out[i,j,:]=x[i*stride:i*stride+size,j*stride:j*stride+size,:].max((0,1))
    return out
# output size:  (W - K + 2P)//S + 1


---
### ✅ Checklist
- [ ] Explain local patterns / translation invariance / spatial hierarchy.
- [ ] Implement conv2d and maxpool from scratch and verify against Keras.
- [ ] Use the output-size formula; know valid vs same padding and what strides do.
- [ ] Build a Conv→Pool→GAP→Dense classifier; know depth↑/size↓.
- [ ] Apply data augmentation for small datasets.
- [ ] Do transfer learning: frozen feature extraction, then fine-tuning at low lr.

**Next: Chapter 9** — *ConvNet architecture patterns*: residual connections, batch normalization, depthwise
separable convolutions — the building blocks of modern vision models. Say "Chapter 9".